In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score,mean_absolute_error, mean_squared_error
from transformers import RobertaTokenizer, RobertaModel
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from torch.utils.data import DataLoader, Subset
from scipy.stats import pearsonr
from tqdm import tqdm
from sklearn.exceptions import FitFailedWarning
import warnings
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV
import matplotlib.pyplot as plt
from scipy.stats import norm
import matplotlib
from sklearn.base import clone


In [2]:
data=pd.read_excel("./invertebrates_unique.xlsx")
data=data.dropna()
smiles_data = data['SMILES_Canonical_RDKit'].tolist()
mgperL = data['mgperL'].values
Duration_Value= data['Duration_Value'].values

mgperL=np.log1p(mgperL)

In [3]:
# 检查需要One-Hot编码的列，并进行编码（如果类别超过一种）
def encode_column(data, column_name):
    unique_values = data[column_name].unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(data[[column_name]])
    else:
        return None  # 只有一种类别时忽略

# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(data, 'effect')
endpoint_encoded = encode_column(data, 'endpoint')
species_encoded = encode_column(data, 'species_group')

# 将需要的列拼接成输入 X
extra_features = data['Duration_Value'].values.reshape(-1, 1)

# 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded, species_encoded]:
    if encoded_feature is not None:
        extra_features = np.hstack((extra_features, encoded_feature))


extra_dim = extra_features.shape[1]
data_extra_features = extra_features

In [4]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import BertTokenizerFast, BertModel
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import optuna

# -------------------------------
# 1. 数据增强与 Dataset 修改：支持额外特征 extra_features
# -------------------------------
def augment_smiles(smiles):
    """简单的数据增强方法：50% 的概率翻转 SMILES 字符串"""
    if random.random() > 0.5:
        return smiles[::-1]
    return smiles

class SMILES_Dataset(Dataset):
    def __init__(self, smiles, reg_labels, extra_features=None, use_augmentation=False):
        self.smiles = smiles
        self.reg_labels = reg_labels
        self.extra_features = extra_features  # 额外特征，要求为数组或列表（每个元素是一个数值向量）
        self.use_augmentation = use_augmentation

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        s = self.smiles[idx]
        if self.use_augmentation:
            s = augment_smiles(s)
        reg_label = self.reg_labels[idx]
        tokens = tokenizer(s, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        # squeeze 扁平化 batch 维度（例如变为 [seq_len]）
        tokens = {key: val.squeeze(0) for key, val in tokens.items()}
        if self.extra_features is not None:
            extra_feat = self.extra_features[idx]
            extra_feat = torch.tensor(extra_feat, dtype=torch.float32)
            return tokens, torch.tensor(reg_label, dtype=torch.float32), extra_feat
        else:
            return tokens, torch.tensor(reg_label, dtype=torch.float32)


In [5]:

# -------------------------------
# 2. 加载本地 BERT 模型（BERT for SMILES）
# -------------------------------
# 请将 checkpoint 指向你的本地模型目录（此处示例使用 'unikei/bert-base-smiles'）
checkpoint = '../models/base_bert'
tokenizer = BertTokenizerFast.from_pretrained(checkpoint)
bert_model = BertModel.from_pretrained(checkpoint)

# -------------------------------
# 3. 模型定义：BERT_Regression
# -------------------------------
class Bert_Regression(nn.Module):
    def __init__(self, dropout_rate, fc1_size, fc2_size, fc3_size, extra_dim=0):
        """
        :param extra_dim: 额外特征维度，如果为 0 则不拼接额外特征
        """
        super(Bert_Regression, self).__init__()
        self.bert = bert_model  # 使用加载好的 BERT 模型
        hidden_size = self.bert.config.hidden_size  # 通常为768
        self.extra_dim = extra_dim
        # 拼接后的维度
        input_dim = hidden_size + extra_dim
        self.regressor = nn.Sequential(
            nn.Linear(input_dim, fc1_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc1_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc1_size, fc2_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc2_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc2_size, fc3_size),
            nn.ReLU(),
            nn.BatchNorm1d(fc3_size),
            nn.Dropout(dropout_rate),
            nn.Linear(fc3_size, 1),
            nn.Softplus()
        )
    
    def forward(self, tokens, extra_features=None):
        outputs = self.bert(**tokens)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token 嵌入，形状 [batch_size, 768]
        if self.extra_dim > 0 and extra_features is not None:
            # 拼接额外特征，要求 extra_features 的形状为 [batch_size, extra_dim]
            x = torch.cat([cls_embedding, extra_features], dim=1)
        else:
            x = cls_embedding
        reg_output = self.regressor(x)
        return reg_output

In [6]:

data_smiles = smiles_data
data_labels = mgperL

# 使用 SMILES 作为组依据，确保同一 SMILES 不出现在不同折中
groups = data_smiles

# 如果数据量较大，可根据需要提前封装为 Dataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [7]:
# 3. 定义嵌入提取函数
def get_embeddings_for_loader(model, loader):
    all_embeddings = []
    all_extra_features = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for tokens, labels_tensor, extra_feats in loader:
            tokens = {key: val.to(device) for key, val in tokens.items()}
            extra_feats = extra_feats.to(device)
            outputs = model.bert(**tokens)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [B, 768]
            all_embeddings.append(cls_embeddings.cpu().numpy())
            all_extra_features.append(extra_feats.cpu().numpy())
            all_labels.append(labels_tensor.numpy())
    return (
        np.vstack(all_embeddings),
        np.vstack(all_extra_features),
        np.concatenate(all_labels)
    )


In [8]:
'''{'dropout_rate': 0.2096365380068255, 
'fc1_size': 960, 
'fc2_size': 288, 
'fc3_size': 128, 
'learning_rate': 1.4090853843658563e-05, 
'weight_decay': 0.020563427953581105}'''

"{'dropout_rate': 0.2096365380068255, \n'fc1_size': 960, \n'fc2_size': 288, \n'fc3_size': 128, \n'learning_rate': 1.4090853843658563e-05, \n'weight_decay': 0.020563427953581105}"

In [9]:
from sklearn.utils import shuffle


from torch.cuda.amp import autocast, GradScaler

# 直接使用之前超参数搜索得到的最优参数（保证与搜索阶段一致）
best_params = {
    'dropout_rate': 0.2096365380068255,
    'fc1_size': 960,
    'fc2_size': 288,
    'fc3_size': 128,
    'learning_rate': 1.4090853843658563e-05,
    'weight_decay': 0.020563427953581105
}

# 从 best_params 中提取各个超参数
best_dropout = best_params['dropout_rate']
best_fc1 = best_params['fc1_size']
best_fc2 = best_params['fc2_size']
best_fc3 = best_params['fc3_size']
best_lr = best_params['learning_rate']
best_wd = best_params['weight_decay']


gkf = GroupKFold(n_splits=10)

for z, (train_idx, val_idx) in enumerate(gkf.split(data_smiles, data_labels, groups)):
    print(f"\nProcessing Fold {z+1}")

    # 获取每折数据
    train_smiles = [data_smiles[i] for i in train_idx]
    val_smiles = [data_smiles[i] for i in val_idx]
    train_labels = [data_labels[i] for i in train_idx]
    val_labels = [data_labels[i] for i in val_idx]
    train_extra = data_extra_features[train_idx]
    val_extra = data_extra_features[val_idx]

    # 构造 Dataset 和 DataLoader
    train_dataset = SMILES_Dataset(train_smiles, train_labels, extra_features=train_extra)
    val_dataset = SMILES_Dataset(val_smiles, val_labels, extra_features=val_extra)
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

    # 初始化模型
    model = Bert_Regression(best_dropout, best_fc1, best_fc2, best_fc3, extra_dim=extra_dim).to(device)

    # 加载该折训练好的模型
    model.load_state_dict(torch.load(f'./optuna_model_set_1/bert_reg_fold_{z+1}.pth', map_location=device))
    model.eval()

    # 提取并拼接 SMILES 嵌入 + 额外特征
    smiles_emb_train, extra_feat_train, labels_train = get_embeddings_for_loader(model, train_loader)
    smiles_emb_val, extra_feat_val, labels_val = get_embeddings_for_loader(model, val_loader)

    X_train = np.hstack([smiles_emb_train, extra_feat_train])
    X_val = np.hstack([smiles_emb_val, extra_feat_val])

    # 保存数据
    np.save(f'./k_folds_model/embeddings/train_fold_{z+1}.npy', X_train)
    np.save(f'./k_folds_model/embeddings/val_fold_{z+1}.npy', X_val)
    np.save(f'./k_folds_model/embeddings/train_labels_fold_{z+1}.npy', labels_train)
    np.save(f'./k_folds_model/embeddings/val_labels_fold_{z+1}.npy', labels_val)

    # =============== ✅ 保存模型预测值 ===============
    y_preds = []

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for tokens, labels_tensor, extra_feats in val_loader:
            tokens = {k: v.to(device) for k, v in tokens.items()}
            extra_feats = extra_feats.to(device)
            labels_tensor = labels_tensor.to(device)
            outputs = model(tokens, extra_feats)
            preds.extend(outputs.squeeze().cpu().numpy())
            targets.extend(labels_tensor.cpu().numpy())
    
    y_preds = np.array(preds)
    # 保存预测值为 pkl（可选：也可以保存为 npy）
    np.save(f'./k_folds_model/bert_prediction/bert_fold_{z+1}_predictions.npy', y_preds)


    print(f"✅ Fold {z+1} embeddings, labels, and predictions saved.")



Processing Fold 1
✅ Fold 1 embeddings, labels, and predictions saved.

Processing Fold 2
✅ Fold 2 embeddings, labels, and predictions saved.

Processing Fold 3
✅ Fold 3 embeddings, labels, and predictions saved.

Processing Fold 4
✅ Fold 4 embeddings, labels, and predictions saved.

Processing Fold 5
✅ Fold 5 embeddings, labels, and predictions saved.

Processing Fold 6
✅ Fold 6 embeddings, labels, and predictions saved.

Processing Fold 7
✅ Fold 7 embeddings, labels, and predictions saved.

Processing Fold 8
✅ Fold 8 embeddings, labels, and predictions saved.

Processing Fold 9
✅ Fold 9 embeddings, labels, and predictions saved.

Processing Fold 10
✅ Fold 10 embeddings, labels, and predictions saved.


In [ ]:
#smiles_embeddings=np.load('./embedding/fish_EC10_smiles_embeddings.npy')